# Components lab 05: memories

A quantum memory is a component with physical positions. It absorbs qstate-backed signals, relabels photon subsystems into stable memory subsystems, emits later photons, and reports every operation.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

from simyuj.components import Port, PortDelivery, PortDirection, PortKind, connect_ports
from simyuj.components.memories import (
    MEMORY_ABSORB,
    MEMORY_APPLY_OPERATOR,
    MEMORY_DISCARD,
    MEMORY_EMIT,
    MEMORY_MEASURE,
    MemoryAbsorbRequest,
    MemoryApplyOperatorRequest,
    MemoryDiscardRequest,
    MemoryEmitRequest,
    MemoryMeasureRequest,
    QuantumMemory,
)
from simyuj.engine import Component, Event, Timeline
from simyuj.primitives.subsystems import SubsystemHandle
from simyuj.qstate import SubsystemId
from simyuj.qstate.noise import depolarizing
from simyuj.qstate.ops import X
from simyuj.runtime.binding import BindingContext
from simyuj.signal import EncodingScheme, Signal, SignalKind

## 1. Sinks and helpers

The notice sink receives memory reports. The quantum sink receives emitted photons.

In [ ]:
@dataclass(slots=True)
class NoticeSink(Component):
    component_id: str
    input_port: Port = field(init=False)
    reports: list[tuple[int, object]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port('in', self, self.component_id, PortKind.CLASSICAL, PortDirection.INGRESS)

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.reports.append((timeline.current_time, delivery.payload))


In [ ]:
@dataclass(slots=True)
class QuantumSink(Component):
    component_id: str
    input_port: Port = field(init=False)
    signals: list[tuple[int, Signal]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port('in', self, self.component_id, PortKind.QUANTUM, PortDirection.INGRESS)

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.signals.append((timeline.current_time, delivery.payload))


In [ ]:
def make_photon(timeline: Timeline, *, signal_id: str, state: str, time: int) -> Signal:
    subsystem = SubsystemId(f'{signal_id}:photon')
    state_ref = timeline.qstate.prepare(state, subsystems=(subsystem,))
    return Signal(
        id=signal_id,
        signal_kind=SignalKind.PHOTON,
        encoding_scheme=EncodingScheme.POLARIZATION,
        emission_time=time,
        origin='source',
        state_ref=state_ref,
        state_targets=(SubsystemHandle(label=str(subsystem), kind='qubit', index=0),),
    )


In [ ]:
def print_positions(memory: QuantumMemory) -> None:
    for position in memory.positions:
        print(
            'position', position.position,
            'status=', position.status.value,
            'ready_at=', position.ready_at,
            'token=', position.occupancy_token,
        )


## 2. Absorb with delay and report notices

A delayed absorb marks the position busy first, then completes later. The incoming photon subsystem is relabeled into a stable memory subsystem at completion.

In [ ]:
timeline = Timeline(master_seed=12)
notice_sink = NoticeSink('memory.notices')
output_sink = QuantumSink('memory.output')

memory = QuantumMemory(
    memory_id='nodeA.mem0',
    num_positions=1,
    absorb_delay_ticks=2,
    emit_delay_ticks=3,
    measure_delay_ticks=1,
    recovery_ticks=2,
    storage_lifetime_ticks=20,
    noise_models=(depolarizing(0.02),),
)
memory.bind(BindingContext(timeline=timeline, logger=timeline.logger))
connect_ports(memory.notice_port, notice_sink.input_port, target_action='memory_notice')
connect_ports(memory.output_port, output_sink.input_port, target_action='receive_emitted_photon')

photon = make_photon(timeline, signal_id='sig-1', state='|+>', time=0)
print('before absorb:')
print_positions(memory)
print('qstate size:', timeline.qstate.size())

In [ ]:
timeline.schedule(
    Event(
        time=0,
        target_ref=memory,
        action=MEMORY_ABSORB,
        payload_ref=MemoryAbsorbRequest(
            request_id='absorb-1',
            memory_id=memory.memory_id,
            signal=photon,
            position=0,
            meta=(('slot', 'left'),),
        ),
    )
)

timeline.run_until(0)
print('after scheduling delayed absorb at t=0:')
print_positions(memory)
print('reports so far:', len(memory.reports))

timeline.run_until(2)
print('\nafter absorb completes at t=2:')
print_positions(memory)
print('latest report:', memory.reports[-1])
print('notice sink:', notice_sink.reports[-1])
record = timeline.qstate.record(timeline.qstate.state_of(SubsystemId('memory:nodeA.mem0:position:0')))
print('stored qstate subsystem:', tuple(str(s) for s in record.layout.subsystems))
print('stored rep:', record.rep)

## 3. Measure and operate on stored positions

Memory operations target physical positions. The component translates those positions into qstate subsystems.

In [ ]:
timeline.schedule(
    Event(
        time=3,
        target_ref=memory,
        action=MEMORY_MEASURE,
        payload_ref=MemoryMeasureRequest(
            request_id='measure-1',
            memory_id=memory.memory_id,
            positions=(0,),
            measurement='x',
            destructive=False,
        ),
    )
)
timeline.run_until(4)
print('measurement report:', memory.reports[-1])
print_positions(memory)

In [ ]:
timeline.schedule(
    Event(
        time=5,
        target_ref=memory,
        action=MEMORY_APPLY_OPERATOR,
        payload_ref=MemoryApplyOperatorRequest(
            request_id='flip-1',
            memory_id=memory.memory_id,
            positions=(0,),
            operator=X,
        ),
    )
)
timeline.run_until(5)
print('operator report:', memory.reports[-1])
print_positions(memory)

## 4. Emit from memory

Emission relabels the memory subsystem into a fresh photon subsystem and clears the physical position into recovery.

In [ ]:
timeline.schedule(
    Event(
        time=6,
        target_ref=memory,
        action=MEMORY_EMIT,
        payload_ref=MemoryEmitRequest(
            request_id='emit-1',
            memory_id=memory.memory_id,
            position=0,
            meta=(('path', 'to-bob'),),
        ),
    )
)

timeline.run_until(6)
print('after emit request starts:')
print_positions(memory)

timeline.run_until(9)
print('\nafter emit completes:')
print_positions(memory)
print('latest report:', memory.reports[-1])
print('emitted signals:', [(time, signal.id, signal.state_targets[0].label) for time, signal in output_sink.signals])
print('qstate size:', timeline.qstate.size())

## 5. Expiry is scheduled by the memory

A stored position with a finite lifetime clears itself when its occupancy token still matches.

In [ ]:
timeline = Timeline(master_seed=3)
notice_sink = NoticeSink('expiry.notices')
memory = QuantumMemory(
    memory_id='nodeA.short_mem',
    num_positions=1,
    storage_lifetime_ticks=5,
)
memory.bind(BindingContext(timeline=timeline, logger=timeline.logger))
connect_ports(memory.notice_port, notice_sink.input_port, target_action='memory_notice')

photon = make_photon(timeline, signal_id='short-lived', state='|0>', time=0)
timeline.schedule(
    Event(
        time=0,
        target_ref=memory,
        action=MEMORY_ABSORB,
        payload_ref=MemoryAbsorbRequest('absorb-short', memory.memory_id, photon, position=0),
    )
)
timeline.run_until(0)
print('after absorb:')
print_positions(memory)
print('qstate size:', timeline.qstate.size())

timeline.run_until(6)
print('\nafter lifetime passes:')
print_positions(memory)
print('latest report:', memory.reports[-1])
print('qstate size:', timeline.qstate.size())

## Keep this model in your head

Memory positions are physical slots. Requests name positions; reports describe outcomes. Qstate subsystem labels change as photons enter and leave memory.